# Analysis for the cosmology benchmark (C functions)

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import sympy
import matplotlib.pyplot as plt

from pathlib import Path

import AnalysisUtils as au

In [ ]:
Path("Results").mkdir(parents=True, exist_ok=True)

In [ ]:
x1, x2 = sympy.symbols("x1:3")
x = sympy.abc.x
y = sympy.abc.y
z = sympy.abc.z

## Loading data

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("For-analysis/full-report.csv")

In [ ]:
full_report.run_set.unique()

In [ ]:
full_report.sort_values(["run_set", "data_set", "mse"], inplace=True)
f_indices = full_report.data_set.apply(lambda x: x.startswith("C"))
fr2 = full_report.loc[f_indices].set_index(["run_set", "data_set", "sample_num"])

In [ ]:
fr2["sympy"] = fr2.expr_original_syms.apply(au.parse_if_needed)
fr2["sympy_defuzz"] = fr2.expr_original_syms_defuzz.apply(au.parse_if_needed)

Out of all the run sets, these are the two that will be analyzed and reported.

In [ ]:
srb_key = "SRB-2026-07-20-1300"
srb_extra_key = "SRB-2026-07-25-1900"
cht_key = "CHT-2026-07-20-1300"

In [ ]:
srb = fr2.loc[srb_key]
cht = fr2.loc[cht_key]

In [ ]:
srb.groupby(level=["data_set"]).size()

These are the best ones overall

In [ ]:
srb_min_mse_ixs = srb.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb.loc[srb_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht_min_mse_ixs = cht.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb_threshold_table = au.mse_threshold_table(srb)
srb_threshold_table

In [ ]:
cht_threshold_table = au.mse_threshold_table(cht)
cht_threshold_table

In [ ]:
au.to_latex(
    srb_threshold_table,
    file="Generated/srb_threshold_table.tex",
    strip_colname_prefix="mse",
)

In [ ]:
au.to_latex(
    cht_threshold_table,
    file="Generated/cht_threshold_table.tex",
    strip_colname_prefix="mse",
)

These are problems I ran extra samples of to get better reliability estimates.

In [ ]:
fr2.loc[([srb_key, srb_extra_key], ["C3h"]),:]

In [ ]:
extra_problems = ["C3h", "C5d"]
srb_extra = fr2.loc[([srb_key, srb_extra_key], extra_problems),:]

In [ ]:
srb_extra.sort_values("mse", ascending=True).head()

## Polynomials

In [ ]:
data_sets_polynomial = ["C2a"]

In [ ]:
srb.loc[data_sets_polynomial]

`C2a` is no problem.

I used to include `C5f` here, but it's a rational function, not a polynomial.
Table on p31 of the cosmology article is confusing, because `C2a` has a reference to $H(z)$ but there's a column of $H$ in the data file and it really is a simple polynomial.
But `C5f`, which looks like the same kind of item, seems to be referring to `C5d` and `C5e` as functions or $R_0$ and $r$.

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_polynomial, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
(srb.loc[data_sets_polynomial]
 .sympy_defuzz.apply(lambda e: not e.is_polynomial(x1, x2))
 .groupby(level="data_set")
 .sum())

All runs on all polynomial data sets are correct up to fuzz.

## Rational functions

In [ ]:
data_sets_rational = ["C3g", "C3h", "C5a", "C5b", "C5c", "C5d", "C5e", "C5f"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C3g: no, but MSE is better than reference
C3h: no, but MSE is better than reference
C5a: perfect
C5b: no
C5c: no
C5d: perfect but in a different form, see below
C5e: no, but MSE is not bad
C5f: no
```

In [ ]:
C5d_best = srb.loc[srb_min_mse_ixs].loc["C5d"].sympy_defuzz.iloc[0]

In [ ]:
C5d_best

In [ ]:
sympy.simplify(sympy.together(C5d_best))

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C3g: no, but MSE is better than reference
C3h: perfect after defuzzing
C5a: perfect
C5b: no
C5c: no
C5d: perfect
C5e: no, but MSE is not bad
C5f: no
```

In [ ]:
C3h_cht_best = cht.loc[cht_min_mse_ixs].loc["C3h"].sympy_defuzz.iloc[0]

In [ ]:
au.replace_near_integer(sympy.expand(C3h_cht_best))

In [ ]:
C5b_cht_best = cht.loc[cht_min_mse_ixs].loc["C5b"].sympy_defuzz.iloc[0]

In [ ]:
sympy.together(C5b_cht_best)

In [ ]:
C5d_cht_best = cht.loc[cht_min_mse_ixs].loc["C5d"].sympy_defuzz.iloc[0]

In [ ]:
sympy.simplify(sympy.together(sympy.expand(C5d_cht_best)))

In [ ]:
C5e_cht_best = cht.loc[cht_min_mse_ixs].loc["C5e"].sympy_defuzz.iloc[0]

In [ ]:
sympy.expand(C5e_cht_best)

The majority of solutions are not rational functions.

In [ ]:
(srb.loc[data_sets_rational]
 .sympy_defuzz
 .apply(lambda e: not e.is_rational_function())
 .groupby(level="data_set").sum())

In [ ]:
rational_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 399),
    "complexity_binwidth": 20,
    "mse_lims": (1.0e-32, 1.e2),
    "mse_binwidth": 2.0,
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_rational],
    file_stem="srb-rational-complexity-mse-displot",
    **rational_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_rational],
    file_stem="cht-rational-complexity-mse-displot",
    **rational_plot_params
)

In [ ]:
cht.loc["C3h"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=1e-3))

### More reliability estimates

In [ ]:
srb.loc["C5a"].sympy.apply(lambda e: au.replace_near_integer(
    sympy.expand(e.subs(x, 1)).evalf(),
    tolerance=2e-6))

In [ ]:
srb.loc["C5b"].sympy.apply(lambda e: au.replace_near_integer(
    sympy.expand(e.subs(x, 1)).evalf(),
    tolerance=2e-6))

In [ ]:
srb.loc["C5c"].sympy.apply(lambda e: au.replace_near_integer(
    sympy.expand(e.subs(x, 1)).evalf(),
    tolerance=2e-6))

C5d includes a constant column of $x = +1$.
C5e includes a constant column of $x = -1$.

In [ ]:
srb.loc["C5d"].sympy.apply(lambda e: au.replace_near_integer(
    sympy.expand(e.subs(x, 1)).evalf(),
    tolerance=0.01))

In [ ]:
srb.loc["C5e"].sympy.apply(lambda e: au.replace_near_integer(
    sympy.expand(e.subs(x, -1)).evalf(),
    tolerance=0.01))

### Extra micro-trials

I ran more samples for a few data sets, so now we have 192.

In [ ]:
srb_extra.shape

In [ ]:
srb_C3h = srb_extra.loc[([srb_key, srb_extra_key], ["C3h"]),:].sort_values("mse")

There are 3 clearly good ones, so the probability of success in 8h is at least 19.7%

In [ ]:
srb_C3h.sympy.iloc[0:10].apply(lambda e: au.replace_near_integer(e.evalf(), tolerance=1.0e-2))

In [ ]:
srb_C5d = srb_extra.loc[([srb_key, srb_extra_key], ["C5d"]),:].sort_values("mse")
srb_C5d

In [ ]:
srb_C5d.sympy.iloc[0:5].apply(
    lambda e:
    au.replace_near_integer(
        sympy.simplify(
            sympy.expand(
                sympy.together(
                    au.replace_near_integer(e.subs(x, 1).evalf(), tolerance=2.0e-3)))),
                    tolerance=1e-8)
        )

## Power functions

In [ ]:
data_sets_power = ["C1a", "C1b", "C1c", "C1d", "C2b"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_power, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C1a: no
C1b: no, although it has a term of roughly the correct form.
C1c: no
C1d: no
C2b: no
````

In [ ]:
C1b_best = srb.loc[srb_min_mse_ixs].loc["C1b"].sympy_defuzz.iloc[0]

In [ ]:
C1b_best

In [ ]:
C1c_best = srb.loc[srb_min_mse_ixs].loc["C1c"].sympy_defuzz.iloc[0]

In [ ]:
C1c_best

In [ ]:
au.replace_near_integer(C1c_best, tolerance=1.0e-4)

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_power, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Kind of the same.

Let's take a look at the C1a data.
It's just a slight curve.

In [ ]:
C1a_df = pd.read_csv("Things-to-bench/cosmo_data/C1a.csv")

In [ ]:
C1a_plot = sns.relplot(data=C1a_df, x="z", y="target", kind="line")

In [ ]:
C1a_plot.savefig("Generated/Pictures/C1a_plot.svg")
C1a_plot.savefig("Generated/Pictures/C1a_plot.pdf")

I bet if we gave it something with more shape it would have better luck.

In [ ]:
xs = np.linspace(-2.2, 2, 100)
ys = 0.0718 * np.sqrt(0.3 * (1 + xs) ** 3 + 0.7)

In [ ]:
sns.lineplot(x=xs, y=ys)

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_power],
    file_stem="srb-power-complexity-mse-displot",
    **rational_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_power],
    file_stem="cht-power-complexity-mse-displot",
    **rational_plot_params
)

## Hard

In [ ]:
data_sets_hard = ["C6a", "C6b", "C6c"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_hard, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C6a: nope
C6b: decent MSE, better than cp3 article, some promising terms
C6c: nope
```

These are the best MSEs from the cp3-bench article.

In [ ]:
targets_C6 = [ 0.0022, 1.2e-5, 0.11]
target_map = dict(zip(data_sets_hard, targets_C6))

This sort of bizarre construction makes an index that can be used as a series of thresholds lined up with the rows of `srb_C6`.

In [ ]:
srb_C6 = srb.loc[data_sets_hard]
threshold_series = srb_C6.index.get_level_values("data_set").map(target_map)
threshold_series

In [ ]:
C6_threshold_counts = (srb_C6.mse <= threshold_series).groupby(level="data_set").sum()
C6_best = srb_C6.mse.groupby(level="data_set").min()

In [ ]:
C6_threshold_table = pd.DataFrame({"best": C6_best, "target": targets_C6, "count": C6_threshold_counts}, index=data_sets_hard)
C6_threshold_table

In [ ]:
C6_threshold_table.to_csv("Results/C6_threshold_table.csv", index=True, header=True)

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_hard, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Nope.

In [ ]:
hard_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 399),
    "complexity_binwidth": 20,
    "mse_lims": (1.0e-10, 1.e3),
    "mse_binwidth": 1.0,
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_hard],
    file_stem="srb-hard-complexity-mse-displot",
    **hard_plot_params
)

## Black box

In [ ]:
data_sets_C3bb = ["C3a", "C3b", "C3c", "C3d", "C3e", "C3f"]
data_sets_C4bb = ["C4a", "C4b", "C4c", "C4d", "C4e"]
data_sets_black_box = data_sets_C3bb + data_sets_C4bb

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_black_box, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In the cp3 article, there's no given exact form for these problems, just an order of magnitude for MSE for the best result.
So the best I can do is compare MSE by the power of 10.
```
C3a: better than cp3
C3b: better than cp3
C3c: similar to cp3
C3d: better than cp3
C3e: similar to cp3
C3f: better than cp3
C4a: better than cp3
C4b: similar to cp3
C4c: similar to cp3
C4d: similar to cp3
C4e: worse than cp3
```

In [ ]:
black_box_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 399),
    "complexity_binwidth": 20,
    "mse_lims": (1.0e-15, 1.e-4),
    "mse_binwidth": 0.5,
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

These are the best MSEs from the cp3-bench article.

In [ ]:
targets_C3bb = [
    4.2e-13,
    3.6e-11,
    1.4e-7,
    1.3e-6,
    4.6e-7,
    3.8e-6,
    ]
targets_C4bb = [
    4.0e-2,
    3.5e-2,
    3.6e-2,
    1.7e-3,
    4.8e-5,
]
targets_bb = targets_C3bb + targets_C4bb
target_map = dict(zip(data_sets_C3bb + data_sets_C4bb, targets_C3bb + targets_C4bb))

In [ ]:
srb.loc[data_sets_black_box, :]

In [ ]:
srb_bb = srb.loc[data_sets_black_box]

In [ ]:
threshold_series = srb_bb.index.get_level_values("data_set").map(target_map)

In [ ]:
bb_threshold_counts = (srb_bb.mse <= threshold_series).groupby(level="data_set").sum()
bb_best = srb_bb.mse.groupby(level="data_set").min()

In [ ]:
bb_threshold_table = pd.DataFrame({"best": bb_best, "target": targets_bb, "count": bb_threshold_counts}, index=data_sets_black_box)
bb_threshold_table

In [ ]:
bb_threshold_table.to_csv("Results/bb_threshold_table.csv", index=True, header=True)

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_C3bb],
    file_stem="srb-C3bb-complexity-mse-displot",
    mse_target=targets_C3bb,
    **black_box_plot_params
)

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_C4bb],
    file_stem="srb-C4bb-complexity-mse-displot",
    mse_target=targets_C4bb,
    **(black_box_plot_params | { "mse_lims": (1e-5, 1.0), "mse_binwidth": 0.2 })
)

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_black_box],
    file_stem="srb-black-box-complexity-mse-displot",
    **black_box_plot_params
)

### More about C3b

Since Jessamine did so much better on C3b, let's see if there's anything to be learned from it.

In [ ]:
srb_C3b = srb.loc["C3b"]
srb_C3b

In [ ]:
srb_C3b.iloc[0].sympy

In [ ]:
sympy.simplify(srb_C3b.iloc[0].sympy_defuzz)

In [ ]:
C3b_expanded = sympy.expand(srb_C3b.iloc[0].sympy_defuzz)
C3b_expanded

In [ ]:
f_C3b = sympy.lambdify(z, srb_C3b.iloc[0].sympy_defuzz)

In [ ]:
C3b_df = pd.read_csv("Things-to-bench/cosmo_data/C3b.csv")

In [ ]:
C3b_df["prediction"] = f_C3b(np.array(C3b_df.z))

In [ ]:
C3b_df

In [ ]:
C3b_pic = sns.relplot(data=C3b_df, x="z", y="target", kind="line")
C3b_pic.savefig("Generated/Pictures/C3b_plot.svg")
C3b_pic.savefig("Generated/Pictures/C3b_plot.pdf")

In [ ]:
sns.relplot(data=C3b_df, x="z", y="prediction", kind="line")

This pulls apart all the terms.

In [ ]:
srb_C3b.iloc[0].sympy.args

In [ ]:
C3b_terms_fns = [sympy.lambdify(z, term) for term in srb_C3b.iloc[0].sympy.args]

In [ ]:
sorted_terms = sorted([(np.linalg.norm(f(C3b_df.z)), j) for f, j in zip(C3b_terms_fns, range(len(C3b_terms_fns)))])
sorted_terms.reverse()
sorted_terms

In [ ]:
[srb_C3b.iloc[0].sympy.args[j] for _, j in sorted_terms]

In [ ]:
sympy.latex(C3b_expanded.evalf(3, chop=True))

In [ ]:
C3b_sorted_latex = [sympy.latex(srb_C3b.iloc[0].sympy.args[j].evalf(3, chop=True)) for _, j in sorted_terms]

In [ ]:
print("\n".join(C3b_sorted_latex))


In [ ]:
C3b_truncated = sympy.Add(*[srb_C3b.iloc[0].sympy.args[j] for j in range(8)])
C3b_truncated

In [ ]:
f_C3b_truncated = sympy.lambdify(z, sympy.Add(*[srb_C3b.iloc[0].sympy.args[j] for j in range(8)]))

In [ ]:
C3b_df["prediction_truncated"] = f_C3b_truncated(np.array(C3b_df.z))

In [ ]:
C3b_df

In [ ]:
sns.relplot(data=C3b_df, x="z", y="prediction_truncated", kind="line")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(C3b_df.z, C3b_df.target, label="target")
ax.plot(C3b_df.z, C3b_df.prediction_truncated, label="pred trunc")
# ax.plot(C3b_df.z, 0.105*np.log(C3b_df.z) + 0.0474*C3b_df.z + 0.53, label="linear+log")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot((1+np.exp(-C3b_df.z))**3, C3b_df.target, label="target")
# ax.plot(C3b_df.z, 0.105*np.log(C3b_df.z) + 0.0474*C3b_df.z + 0.53, label="linear+log")

In [ ]:
sympy.series(1 + sympy.exp(-z), z, 0, 4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot((2 - C3b_df.z + 0.5*C3b_df.z**2)**3, C3b_df.target, label="target")
# ax.plot(C3b_df.z, 0.105*np.log(C3b_df.z) + 0.0474*C3b_df.z + 0.53, label="linear+log")

### Cheating with progressive inventory

Cheating isn't really possible because we don't know what hint to give.
Using the progressive inventories:

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_black_box, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_C3bb],
    file_stem="cht-C3bb-complexity-mse-displot",
    mse_target=targets_C3bb,
    **black_box_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_C4bb],
    file_stem="cht-C4bb-complexity-mse-displot",
    mse_target=targets_C4bb,
    **(black_box_plot_params | { "mse_lims": (1e-5, 1.0), "mse_binwidth": 0.2 })
)